# Model 4: Feature Fusion — HashingVectorizer + SentenceTransformer

**Architecture:** Dual-tower fusion combining lexical and semantic features  

```
HashingVec(5000) → LayerNorm → Linear(512) → ReLU ─┐
                                                      ├→ concat(1024) → 4×ResidualBlock → price
SentTrans(384 frozen) → LayerNorm → Linear(512) → ReLU ─┘
```

**Hypothesis:**  
- HashingVec captures *lexical* price signals: brand names ("samsung", "bose"), keywords ("wireless", "4k")  
- SentTrans captures *semantic* price signals: style/quality context ("premium", "economy grade")  
- Fusion of both should outperform either alone

**Note:** SentTrans encoder stays frozen. Only fusion DNN is trained.

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

In [ ]:
from pricer.items import Item
from pricer.fusion_model import FusionRunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

Pre-computes both HashingVec features and SentTrans embeddings once (frozen encoder).  
This step takes ~10-15 min for 800k samples (SentTrans encoding is the bottleneck).

In [ ]:
runner = FusionRunner(train, val[:1000])
runner.setup(batch_size=256)

## 3. Train

Max 15 epochs, early stopping patience=3. CosineAnnealingLR.

In [ ]:
history = runner.train(epochs=15, patience=3)

## 4. Training History

In [ ]:
plot_training_history(history, title="Feature Fusion (HashingVec + SentTrans)")

## 5. Evaluate on 200 Test Samples

In [ ]:
evaluate(runner.inference, test)

## 6. Save Model Weights

In [ ]:
runner.save("fusion_model.pth")
print("Saved to fusion_model.pth")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")